# 17 Bags of N-grams
Example:
- Ori: Dhaval sat on a sofa and ate
- BoW: Dhaval, sat, on, a, sofa, and, ate
- bi-gram: Dhaval sat, sat on, on a, ...
- tri-gram: Dhaval sat on, sat on a, ...

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

v = CountVectorizer(ngram_range=(1,3))
v.fit(["Thor Hathodawala is looking for a job"])
v.vocabulary_

{'thor': 12,
 'hathodawala': 2,
 'is': 5,
 'looking': 9,
 'for': 0,
 'job': 8,
 'thor hathodawala': 13,
 'hathodawala is': 3,
 'is looking': 6,
 'looking for': 10,
 'for job': 1,
 'thor hathodawala is': 14,
 'hathodawala is looking': 4,
 'is looking for': 7,
 'looking for job': 11}

In [6]:
corpus = [
    "Thor ate pizza",
    "Loki is tall",
    "Loki is eating pizza"
]

In [9]:
import spacy

nlp = spacy.load("en_core_web_sm")

def preprocess(text):
  doc = nlp(text)

  filtered_tokens = []

  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filtered_tokens.append(token.lemma_)

  return ' '.join(filtered_tokens)

In [11]:
corpus_processed = [preprocess(text) for text in corpus]

In [12]:
corpus_processed

['thor eat pizza', 'Loki tall', 'Loki eat pizza']

In [16]:
v = CountVectorizer(ngram_range=(1,2))
v.fit(corpus_processed)
v.vocabulary_

{'thor': 7,
 'eat': 0,
 'pizza': 5,
 'thor eat': 8,
 'eat pizza': 1,
 'loki': 2,
 'tall': 6,
 'loki tall': 4,
 'loki eat': 3}

In [19]:
v.transform(["Thor eat pizza"]).toarray()

array([[1, 1, 0, 0, 0, 1, 0, 1, 1]])

In [20]:
import pandas as pd

df = pd.read_json("news_dataset.json")
print(df.shape)

df.head()

(12695, 2)


,text,category
0,Watching Schrödinger's Cat Die University of C...,SCIENCE
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,SCIENCE
2,Entrepreneurs Today Don't Need a Big Budget to...,BUSINESS
3,These Roads Could Recharge Your Electric Car A...,BUSINESS
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,CRIME


In [21]:
df.category.value_counts()

,count
category,
BUSINESS,4254
SPORTS,4167
CRIME,2893
SCIENCE,1381


In [24]:
# undersampling
min_sample = 1381

df_business = df[df.category == "BUSINESS"].sample(min_sample, random_state=42)
df_sports = df[df.category == "SPORTS"].sample(min_sample, random_state=42)
df_crime = df[df.category == "CRIME"].sample(min_sample, random_state=42)
df_science = df[df.category == "SCIENCE"].sample(min_sample, random_state=42)


In [25]:
df_balanced = pd.concat([df_business, df_sports, df_crime, df_science])
df_balanced.category.value_counts()

,count
category,
BUSINESS,1381
SPORTS,1381
CRIME,1381
SCIENCE,1381


In [27]:
target = {"BUSINESS": 0, "SPORTS": 1, "CRIME": 2, "SCIENCE": 3}

df_balanced['category_num'] = df_balanced.category.map(target)
df_balanced.category_num.value_counts()

,count
category_num,
0,1381
1,1381
2,1381
3,1381


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_balanced.text,
    df_balanced.category_num,
    test_size=0.2,
    random_state=42,
    stratify=df_balanced.category_num
)

In [32]:
print(X_train.shape)
X_train.head()

(4419,)


,text
6414,Arby's Employee Keeps Job After Refusing To Se...
1318,Colorful NASA Image Shows Off Pluto’s Psychede...
4170,"Women in Business Q&A: Sophie Delafontaine, Ar..."
11310,5 Formalized Referral Systems to Grow Your Sal...
4188,Hawaii's Kilauea Volcano Sees A Mesmerizing Ri...


In [34]:
y_test.value_counts()

,count
category_num,
2,277
0,276
1,276
3,276


In [41]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('nb', MultinomialNB()),
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.73      0.95      0.82       276
           1       0.92      0.83      0.87       276
           2       0.91      0.88      0.89       277
           3       0.93      0.79      0.85       276

    accuracy                           0.86      1105
   macro avg       0.87      0.86      0.86      1105
weighted avg       0.87      0.86      0.86      1105



In [42]:
from sklearn.ensemble import RandomForestClassifier

clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('rdm', RandomForestClassifier(n_estimators=50)),
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.82      0.74       276
           1       0.87      0.71      0.78       276
           2       0.86      0.85      0.86       277
           3       0.76      0.74      0.75       276

    accuracy                           0.78      1105
   macro avg       0.79      0.78      0.78      1105
weighted avg       0.79      0.78      0.78      1105



In [43]:
# Try preprocess text before modeling
df_balanced["preprocessed_text"] = df_balanced.text.apply(preprocess)

In [ ]:
df_balanced.head()

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced.preprocessed_text,
    df_balanced.category_num,
    test_size=0.2,
    random_state=42,
    stratify=df_balanced.category_num
)

In [45]:
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('nb', MultinomialNB()),
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87       276
           1       0.92      0.87      0.89       276
           2       0.88      0.94      0.91       277
           3       0.93      0.82      0.87       276

    accuracy                           0.89      1105
   macro avg       0.89      0.89      0.89      1105
weighted avg       0.89      0.89      0.89      1105



# Bag of n_grams: Exercise
- Fake news refers to misinformation or disinformation in the country which is spread through word of mouth and more recently through digital communication such as What's app messages, social media posts, etc.

- Fake news spreads faster than Real news and creates problems and fear among groups and in society.

- We are going to address these problems using classical NLP techniques and going to classify whether a given message/ text is Real or Fake Message.

- You will use a Bag of n-grams to pre-process the text and apply different classification algorithms.

- Sklearn CountVectorizer has the inbuilt implementations for Bag of Words.

## About Data: Fake News Detection
- Credits: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

- This data consists of two columns. - Text - label

- Text is the statements or messages regarding a particular event/situation.

- label feature tells whether the given Text is Fake or Real.

- As there are only 2 classes, this problem comes under the Binary Classification.

In [1]:
#import pandas library
import pandas as pd

#read the dataset with name "Fake_Real_Data.csv" and store it in a variable df
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")
fake['label'] = 'fake'
true['label'] = 'true'

#print the shape of dataframe
df = pd.concat([fake, true])
print(df.shape)

#print top 5 rows
df.head()

(44898, 5)


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",fake
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",fake
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",fake
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",fake
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",fake


In [2]:
#check the distribution of labels
df.label.value_counts()

,count
label,
fake,23481
true,21417


In [3]:
# Undersampling
min_samples = 21417

df_fake = df[df.label == "fake"].sample(min_samples, random_state=42)
df_true = df[df.label == "true"].sample(min_samples, random_state=42)

df_final = pd.concat([df_fake, df_true])

In [4]:
df_final.label.value_counts()

,count
label,
fake,21417
true,21417


In [5]:
#Add the new column "label_num" which gives a unique number to each of these labels
label_dict = {'fake': 0, 'true': 1}

df_final['label_num'] = df_final.label.map(label_dict)

#check the results with top 5 rows
df_final.head()

,title,text,subject,date,label,label_num
13474,ABOUT HILLARY’S COUGH: We Discovered The Secre...,,politics,"Jul 20, 2016",fake,0
11994,BREAKING: OBAMACARE REPEAL Clears First Hurdle...,The Senate voted 51-48 this afternoon to proce...,politics,"Jan 4, 2017",fake,0
19179,‘SLEEPY’ JUSTICE GINSBURG: Excites Crowd By Sa...,So much for the SCOTUS not being political Che...,left-news,"Feb 7, 2017",fake,0
501,WATCH: Kellyanne Conway Very Upset Hillary Cl...,White House counselor Kellyanne Conway crawled...,News,"August 24, 2017",fake,0
3492,"GOP Gives Trump The Middle Finger, Prepares T...",Donald Trump may have decided that Russia is g...,News,"December 9, 2016",fake,0


**Modelling without Pre-processing Text data**

In [6]:
#import train-test-split from sklearn
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df_final.text,
    df_final.label_num,
    test_size=0.2,
    random_state=42,
    stratify=df_final.label
)

#Do the 'train-test' splitting with test size of 20% with random state of 2022 and stratify sampling too

In [7]:
y_train.value_counts()

,count
label_num,
1,17134
0,17133


In [8]:
print(X_train.shape)
print(X_test.shape)

(34267,)
(8567,)


**Attempt 1 :**

- using sklearn pipeline module create a classification pipeline to classify the Data.

**Note:**

- using CountVectorizer with unigram, bigram, and trigrams.
- use KNN as the classifier with n_neighbors of 10 and metric as 'euclidean' distance.
- print the classification report.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

#1. create a pipeline object
knn_clf = Pipeline([
    ('vectoerizer', CountVectorizer(ngram_range=(1,3))),
    ('knn', KNeighborsClassifier(n_neighbors=10, metric="euclidean"))
])


#2. fit with X_train and y_train
knn_clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = knn_clf.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.74      0.81      0.77      4284
           1       0.79      0.71      0.75      4283

    accuracy                           0.76      8567
   macro avg       0.76      0.76      0.76      8567
weighted avg       0.76      0.76      0.76      8567



**Attempt 2 :**

- using the sklearn pipeline module create a classification pipeline to classify the Data.

**Note:**

- using CountVectorizer with unigram, bigram, and trigrams.
- use KNN as the classifier with n_neighbors of 10 and metric as 'cosine' distance.
- print the classification report.

In [72]:
#1. create a pipeline object
knn_clf2 = Pipeline([
    ('vectoerizer', CountVectorizer(ngram_range=(1,3))),
    ('knn', KNeighborsClassifier(n_neighbors=10, metric="cosine"))
])


#2. fit with X_train and y_train
knn_clf2.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = knn_clf2.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.98      0.76      4284
           1       0.96      0.41      0.57      4283

    accuracy                           0.69      8567
   macro avg       0.79      0.69      0.67      8567
weighted avg       0.79      0.69      0.67      8567



**Attempt 3 :**

- using the sklearn pipeline module create a classification pipeline to classify the Data.

**Note:**

- using CountVectorizer with only trigrams.
- use RandomForest as the classifier.
- print the classification report.

In [12]:
from sklearn.ensemble import RandomForestClassifier

#1. create a pipeline object
rdm_clf = Pipeline([
    ('vectoerizer', CountVectorizer(ngram_range=(3,3))),
    ('rdm', RandomForestClassifier(n_estimators=10))
])


#2. fit with X_train and y_train
rdm_clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = rdm_clf.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      4284
           1       0.96      0.96      0.96      4283

    accuracy                           0.96      8567
   macro avg       0.96      0.96      0.96      8567
weighted avg       0.96      0.96      0.96      8567



**Attempt 4 :**

- using the sklearn pipeline module create a classification pipeline to classify the Data.

**Note:**

- using CountVectorizer with both unigram and bigrams.
- use Multinomial Naive Bayes as the classifier with an alpha value of 0.75.
- print the classification report.

In [13]:
from sklearn.naive_bayes import MultinomialNB

#1. create a pipeline object
nb_clf = Pipeline([
    ('vectoerizer', CountVectorizer(ngram_range=(3,3))),
    ('nb', MultinomialNB(alpha=0.75))
])


#2. fit with X_train and y_train
nb_clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = nb_clf.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.95      0.97      4284
           1       0.95      0.99      0.97      4283

    accuracy                           0.97      8567
   macro avg       0.97      0.97      0.97      8567
weighted avg       0.97      0.97      0.97      8567



**Use text pre-processing to remove stop words, punctuations and apply lemmatization**

In [14]:
#use this utility function to get the preprocessed text data

import spacy

# load english language model and create nlp object from it
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    # remove stop words and lemmatize the text
    doc = nlp(text)
    filtered_tokens = []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_tokens.append(token.lemma_)

    return " ".join(filtered_tokens)

In [ ]:
# create a new column "preprocessed_txt" and use the utility function above to get the clean data
# this will take some time, please be patient

df_final['preprocessed_txt'] = df_final.text.apply(preprocess)

In [ ]:
# print the top 5 row
df_final.head()

,title,text,subject,date,label,label_num,preprocessed_txt
13474,ABOUT HILLARY’S COUGH: We Discovered The Secre...,,politics,"Jul 20, 2016",fake,0,
11994,BREAKING: OBAMACARE REPEAL Clears First Hurdle...,The Senate voted 51-48 this afternoon to proce...,politics,"Jan 4, 2017",fake,0,Senate vote 51 48 afternoon proceed resolution...
19179,‘SLEEPY’ JUSTICE GINSBURG: Excites Crowd By Sa...,So much for the SCOTUS not being political Che...,left-news,"Feb 7, 2017",fake,0,scotus political check comment equality woman ...
501,WATCH: Kellyanne Conway Very Upset Hillary Cl...,White House counselor Kellyanne Conway crawled...,News,"August 24, 2017",fake,0,White House counselor Kellyanne Conway crawl c...
3492,"GOP Gives Trump The Middle Finger, Prepares T...",Donald Trump may have decided that Russia is g...,News,"December 9, 2016",fake,0,Donald Trump decide Russia go America s new bf...
